# Analyzing Health Data using Support Vector Models

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.pyplot import subplots, cm
import sklearn.model_selection as skm
from ISLP import load_data, confusion_table
from sklearn.preprocessing import OneHotEncoder, LabelBinarizer, StandardScaler
scaler = StandardScaler()
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.svm import SVC, LinearSVC
from ISLP.svm import plot as plot_svm
from sklearn.metrics import RocCurveDisplay
roc_curve = RocCurveDisplay.from_estimator 
import seaborn as sns
from itertools import combinations

## Data Cleaning

In [ ]:
nihs = pd.read_csv("nhis_2022.csv")

In [ ]:
# Removing Responses in Cancer Col which had no answer since I'm focusing on cancer
# It wouldn't make sense to keep non-responses

# There were 7464 responses of 'NIU'
# There were 28 responses of 'refused'
# There were 9 responses of 'don't know'
# This relates to 7501 / 35115 of the data, or 21%

target = 'HEARTATTEV'
nihs = nihs.drop(nihs[(nihs[target] == 0) | (nihs[target] > 6)].index)

# 1 is no cancer
# 2 is yes cancer
# Recode to 0 and 1 for better colormap behavior

nihs[target] = nihs[target].replace({2: 1, 1: 0})

print(nihs[target].value_counts())

In [ ]:
for col in cancer.columns:
    val_counts = cancer[col].value_counts()
    print(f"Column: {col}")
    print(val_counts)
    print("\n") 

In [ ]:
demographics = nihs[['YEAR', 'SERIAL', 'STRATA', 'PSU', 'NHISHID', 'REGION', 'PERNUM', 
                    'NHISPID', 'HHX', 'SAMPWEIGHT', 'ASTATFLG', 'CSTATFLG', 'AGE', 'SEX', 
                    'MARSTCUR', 'EDUC', 'HOURSWRK', 'POVERTY']]

health = nihs[['HEIGHT', 'WEIGHT', 'BMICALC', 'HINOTCOVE', 'CANCEREV', 'CHEARTDIEV', 
                'DIABETICEV', 'HEARTATTEV', 'STROKEV']]

lifestyle =  nihs[['DIABETICEV','ALCANYNO', 'ALCDAYSYR', 'CIGDAYMO', 'MOD10DMIN', 'VIG10DMIN', 'FRUTNO', 
                    'VEGENO', 'JUICEMNO', 'SALADSNO', 'BEANNO', 'SALSAMNO', 'SODAPNO', 
                    'FRIESPNO', 'SPORDRMNO', 'FRTDRINKMNO', 'COFETEAMNO', 'POTATONO', 
                    'PIZZANO', 'HRSLEEP', 'CVDSHT']]

# Examining Feature Interactions

In [ ]:
# I looked at a number of subsets of the data, changing the non-answer cutoff as needed
all_features = nihs[['HHX', 'AGE','HEIGHT', 'WEIGHT', 'BMICALC', 'HINOTCOVE', 'CANCEREV', 'CHEARTDIEV', 
                'HEARTATTEV', 'STROKEV', 'DIABETICEV','ALCANYNO', 'ALCDAYSYR', 'CIGDAYMO',
                 'MOD10DMIN', 'VIG10DMIN', 'FRUTNO', 'VEGENO', 'JUICEMNO', 'SALADSNO', 'BEANNO', 
                 'SALSAMNO', 'SODAPNO', 'FRIESPNO', 'SPORDRMNO', 'FRTDRINKMNO', 'COFETEAMNO', 'POTATONO', 
                    'PIZZANO', 'HRSLEEP']]

# Drop non-numeric or identifier columns
non_features = ['YEAR', 'SERIAL', 'STRATA', 'PSU', 'NHISHID', 'NHISPID', 'HHX', 'PERNUM', 'SAMPWEIGHT']
all_features = all_features.drop(columns=non_features, errors='ignore')

target = 'HEARTATTEV'

# Keep only numeric columns and drop rows with missing values
all_features = all_features.select_dtypes(include = np.number)
all_features = all_features.dropna()

# Only keep rows that have the target variable
all_features = all_features.drop(all_features[(all_features[target] == 0) | (all_features[target] > 6)].index)
all_features[target] = all_features[target].replace({2: 1, 1: 0})

In [ ]:
# Non-answer cutoff point
all_features[all_features > 500] = 0

# Loop through feature combinations to plot
features = [col for col in all_features.columns if col != target]

for x, y in combinations(features, 2):
    plt.figure(figsize = (10, 10))
    plt.scatter(all_features[x], all_features[y], 
                c = all_features[target], cmap = 'coolwarm', alpha = 0.6)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.title(f'{x} vs {y} colored by {target}')
    plt.colorbar(label = target)
    plt.tight_layout()
    plt.show()

## Support Vector Classifier - SCV - Linear

### Age vs Weight

#### Test/Train Split

In [ ]:
age_weight = nihs[[target, 'AGE', 'WEIGHT']]
age_weight = age_weight.drop(age_weight[(age_weight['AGE'] >= 130) | (age_weight['WEIGHT'] >= 400)].index)
train_data_aw, test_data_aw = train_test_split(age_weight, train_size = 0.8, random_state = 42)

X_train_aw = train_data_aw.drop(columns = [target])
y_train_aw = train_data_aw[target]
X_test_aw = test_data_aw.drop(columns = [target])
y_test_aw = test_data_aw[target]

X_train_scaled_aw = scaler.fit_transform(X_train_aw)
X_test_scaled_aw = scaler.transform(X_test_aw)

In [ ]:
print(y_train_aw.value_counts())
print(y_test_aw.value_counts())

In [ ]:
# Creating the initial linear model
svm_linear_aw = SVC(C = 25, kernel = 'linear', class_weight = 'balanced')
svm_linear_aw.fit(X_train_scaled_aw, y_train_aw)

In [ ]:
# Plotting
fig, ax = subplots(figsize = (10, 10))
plot_svm(X_train_scaled_aw,
         y_train_aw.to_numpy(),
         svm_linear_aw,
         ax=ax)

In [ ]:
# Tuning

kfold = skm.KFold(5, random_state = 5, shuffle = True)
grid = skm.GridSearchCV(svm_linear_aw,
                        {'C':[ 0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10, 50]},
                        refit = True,
                        cv = kfold,
                        scoring = 'accuracy')
grid.fit(X_test_scaled_aw, y_test_aw)

print(grid.best_params_)
print(grid.cv_results_[('mean_test_score')])

In [ ]:
# Reruning with tuned parameters
svm_linear_aw2 = SVC(C = 1, kernel = 'linear', class_weight = 'balanced')
svm_linear_aw2.fit(X_train_scaled_aw, y_train_aw)

In [ ]:
# New plot
fig, ax = subplots(figsize = (10, 10))
plot_svm(X_train_scaled_aw,
         y_train_aw.to_numpy(),
         svm_linear_aw2,
         ax=ax)

In [ ]:
y_test_hat_aw2 = svm_linear_aw2.predict(X_test_scaled_aw)
confusion_table(y_test_hat_aw2, y_test_aw)

### Age vs Cigarette Use

In [ ]:
age_cig = nihs[[target, 'AGE', 'CIGDAYMO']]
age_cig = age_cig.drop(age_cig[(age_cig['AGE'] >= 130) | (age_cig['CIGDAYMO'] >= 32)].index)
train_data_ac, test_data_ac = train_test_split(age_cig, train_size = 0.8, random_state = 42)

X_train_ac = train_data_ac.drop(columns = [target])
y_train_ac = train_data_ac[target]
X_test_ac = test_data_ac.drop(columns = [target])
y_test_ac = test_data_ac[target]

X_train_scaled_ac = scaler.fit_transform(X_train_ac)
X_test_scaled_ac = scaler.transform(X_test_ac)

In [ ]:
svm_linear_ac = SVC(C = 5, kernel = 'linear', class_weight = 'balanced')
svm_linear_ac.fit(X_train_scaled_ac, y_train_ac)

In [ ]:
fig, ax = subplots(figsize = (10, 10))
plot_svm(X_train_scaled_ac,
         y_train_ac.to_numpy(),
         svm_linear_ac,
         ax = ax)

In [ ]:
# Tuning
kfold = skm.KFold(5, random_state = 5, shuffle = True)
grid = skm.GridSearchCV(svm_linear_ac,
                        {'C':[10, 1, 0.01, 0.1, 0.001, 0.0001]},
                        refit = True,
                        cv = kfold,
                        scoring = 'accuracy')
grid.fit(X_test_ac, y_test_ac)
print(grid.best_params_)
print(grid.cv_results_[('mean_test_score')])

In [ ]:
# New model with adjusted params
svm_linear_ac2 = SVC(C = 0.1, kernel = 'linear', class_weight = 'balanced')
svm_linear_ac2.fit(X_train_scaled_ac, y_train_ac)

In [ ]:
fig, ax = subplots(figsize = (10, 10))
plot_svm(X_train_scaled_ac,
         y_train_ac.to_numpy(),
         svm_linear_ac2,
         ax = ax)

### Age vs Vigorous Exercise

In [ ]:
age_exc = nihs[[target, 'AGE', 'VIG10DMIN']]
age_exc = age_exc.drop(age_exc[(age_exc['AGE'] >= 130) | (age_exc['VIG10DMIN'] >= 150)].index)
train_data_ae, test_data_ae = train_test_split(age_exc, train_size = 0.8, random_state = 42)

X_train_ae = train_data_ae.drop(columns = [target])
y_train_ae = train_data_ae[target]
X_test_ae = test_data_ae.drop(columns = [target])
y_test_ae = test_data_ae[target]

X_train_scaled_ae = scaler.fit_transform(X_train_ae)
X_test_scaled_ae = scaler.transform(X_test_ae)

In [ ]:
svm_linear_ae = SVC(C = 5, kernel = 'linear', class_weight = 'balanced')
svm_linear_ae.fit(X_train_scaled_ae, y_train_ae)

In [ ]:
fig, ax = subplots(figsize = (10, 10))
plot_svm(X_train_scaled_ae,
         y_train_ae.to_numpy(),
         svm_linear_ae,
         ax = ax)

In [ ]:
# Tuning
kfold = skm.KFold(5, random_state = 5, shuffle = True)
grid = skm.GridSearchCV(svm_linear_ae,
                        {'C':[10, 1, 0.1, 0.01, 0.001, 0.0001]},
                        refit = True,
                        cv = kfold,
                        scoring = 'accuracy')
grid.fit(X_test_ae, y_test_ae)
print(grid.best_params_)
print(grid.cv_results_[('mean_test_score')])

y_test_hat_ae = svm_linear_ae.predict(X_test_ae)
confusion_table(y_test_hat_ae, y_test_ae)

In [ ]:
svm_linear_ae2 = SVC(C = 0.1, kernel = 'linear', class_weight = 'balanced')
svm_linear_ae2.fit(X_train_scaled_ae, y_train_ae)

In [ ]:
fig, ax = subplots(figsize = (10, 10))
plot_svm(X_train_scaled_ae,
         y_train_ae.to_numpy(),
         svm_linear_ae2,
         ax = ax)

In [ ]:
y_test_hat_ae = svm_linear_ae2.predict(X_test_ae)
confusion_table(y_test_hat_ae, y_test_ae)

## ROC Curves - Radial

### Age vs Weight

In [ ]:
age_weight = nihs[[target, 'AGE', 'WEIGHT']]
age_weight = age_weight.drop(age_weight[(age_weight['AGE'] >= 130) | (age_weight['WEIGHT'] >= 400)].index)
train_data_aw, test_data_aw = train_test_split(age_weight, train_size = 0.8, random_state = 42)

X_train_aw = train_data_aw.drop(columns = [target])
y_train_aw = train_data_aw[target]
X_test_aw = test_data_aw.drop(columns = [target])
y_test_aw = test_data_aw[target]

X_train_scaled_aw = scaler.fit_transform(X_train_aw)
X_test_scaled_aw = scaler.transform(X_test_aw)

In [ ]:
svm_rbf_aw = SVC(C = 25, kernel = 'rbf', class_weight = 'balanced', gamma = 1)
svm_rbf_aw.fit(X_train_scaled_aw, y_train_aw)

#### Plot

In [ ]:
fig, ax = subplots(figsize = (6, 6))
plot_svm(X_train_sample,
         y_train_sample,
         svm_rbf_aw,
         ax = ax) 

In [ ]:
svm_rbf_aw = SVC(C = 1, kernel = 'rbf', class_weight = 'balanced', gamma = 1)
svm_rbf_aw.fit(X_train_scaled_aw, y_train_aw)

kfold = skm.KFold(3, random_state = 0,
                  shuffle = True)
grid = skm.GridSearchCV(svm_rbf_aw,
                        {'C':[5, 10],
                         'gamma': [1, 2, 3]},
                        refit = True,
                        cv = kfold,
                        scoring = 'accuracy');
grid.fit(X_train_aw, y_train_aw)
print(grid.best_params_)
print(grid.cv_results_[('mean_test_score')])

In [ ]:
svm_rbf_aw = SVC(C = 50, kernel = 'rbf', class_weight = 'balanced', gamma = 2)
svm_rbf_aw.fit(X_train_scaled_aw, y_train_aw)
fig, ax = subplots(figsize = (8, 8))
plot_svm(X_train_aw,
         y_train_aw,
         best_svm,
         ax = ax)

y_hat_test_aw = best_svm.predict(X_test_aw)
confusion_table(y_hat_test_aw, y_test_aw)


## SVM with Multiple Classes - Polynomial Kernels

### BMI vs Soda Pop Consumption

In [ ]:
bmi_pop  = nihs[[target, 'BMICALC', 'SODAPNO']]
bmi_pop  = bmi_pop .drop(bmi_pop [(bmi_pop ['BMICALC'] >= 60) | (bmi_pop ['SODAPNO'] >= 32)].index)
train_data_bp, test_data_bp = train_test_split(bmi_pop , train_size = 0.8, random_state = 42)

X_train_bp = train_data_bp.drop(columns = [target])
y_train_bp = train_data_bp[target]
X_test_bp = test_data_bp.drop(columns = [target])
y_test_bp = test_data_bp[target]

X_train_scaled_bp = scaler.fit_transform(X_train_bp)
X_test_scaled_bp = scaler.transform(X_test_bp)

In [ ]:
# Diabetes has three classes, (no, yes, borderline) (1, 2, 3)
svm_poly = SVC(kernel = "poly",
                C = 1,
                degree = 2);
svm_poly.fit(X_train_scaled_bp, y_train_bp)
fig, ax = subplots(figsize = (8, 8))
plot_svm(X_train_scaled_bp, y_train_bp, svm_poly, scatter_cmap = cm.tab10, ax = ax)

In [ ]:
kfold = skm.KFold(3, 
                  random_state = 0,
                  shuffle = True)
grid = skm.GridSearchCV(svm_poly,
                        {'C':[0.01, 0.1, 1, 10],
                         'degree':[1, 2, 3]},
                        refit = True,
                        cv = kfold,
                        scoring = 'accuracy');
grid.fit(X_train_scaled_bp, y_train_bp)
grid.best_params_
